In [1]:
import numpy as np
from numba import cuda

cuda.gpus
cuda.detect()
cuda.select_device(0)

Found 1 CUDA devices
id 0                Tesla T4                              [SUPPORTED]
                      Compute Capability: 7.5
                           PCI Device ID: 4
                              PCI Bus ID: 0
                                    UUID: GPU-31d6a172-a53f-e064-dbc7-31d7970a2c54
                                Watchdog: Disabled
             FP32/FP64 Performance Ratio: 32
                            FP16 Support: Yes
                            BF16 Support: No
                             FP8 Support: No
Summary:
	1/1 devices are supported


<weakproxy at 0x780aa138ba60; to 'numba.cuda.cudadrv.driver.Device' at 0x780a956538c0>

In [2]:
from numba import cuda
cc_cores_per_SM_dict = {  (2,0) : 32,  (2,1) : 48, (3,0) : 192, (3,5) : 192, (3,7) : 192, (5,0) : 128, (5,2) : 128, (6,0) : 64, (6,1) : 128, (7,0) : 64, (7,5) : 64, (8,0) : 64, (8,6) : 128, (8,9) : 128, (9,0) : 128 }

device = cuda.get_current_device()
sms = getattr(device, "MULTIPROCESSOR_COUNT")
cc = device.compute_capability
cores_por_sm = cc_cores_per_SM_dict.get(cc)
cores_totales = cores_por_sm * sms
print("GPU compute capability: ", cc)
print("Número de SM: ", sms)
print("Número de cores: ", cores_totales)


GPU compute capability:  ComputeCapability(major=7, minor=5)
Número de SM:  40
Número de cores:  2560


In [3]:
from numba import cuda

device = cuda.get_current_device()

attributes = [
    "WARP_SIZE", "MAX_THREADS_PER_BLOCK",
    "MAX_BLOCK_DIM_X", "MAX_BLOCK_DIM_Y", "MAX_BLOCK_DIM_Z",
    "MAX_GRID_DIM_X", "MAX_GRID_DIM_Y", "MAX_GRID_DIM_Z",
    "MULTIPROCESSOR_COUNT", "MAX_THREADS_PER_MULTIPROCESSOR",
    "MAX_SHARED_MEMORY_PER_BLOCK", "MAX_SHARED_MEMORY_PER_MULTIPROCESSOR",
    "MAX_REGISTERS_PER_BLOCK", "MAX_REGISTERS_PER_MULTIPROCESSOR",
    "CLOCK_RATE",  "MEMORY_CLOCK_RATE", "GLOBAL_MEMORY_BUS_WIDTH", "L2_CACHE_SIZE", ]

print(f"GPU: {device.name}")
print(f"Compute capability: {device.compute_capability}")
print("-" * 55)

for attr in attributes:
    try:
        value = getattr(device, attr)
        print(f"{attr:45s} = {value}")
    except (AttributeError, NotImplementedError):
        print(f"{attr:45s} = NOT SUPPORTED")


GPU: Tesla T4
Compute capability: ComputeCapability(major=7, minor=5)
-------------------------------------------------------
WARP_SIZE                                     = 32
MAX_THREADS_PER_BLOCK                         = 1024
MAX_BLOCK_DIM_X                               = 1024
MAX_BLOCK_DIM_Y                               = 1024
MAX_BLOCK_DIM_Z                               = 64
MAX_GRID_DIM_X                                = 2147483647
MAX_GRID_DIM_Y                                = 65535
MAX_GRID_DIM_Z                                = 65535
MULTIPROCESSOR_COUNT                          = 40
MAX_THREADS_PER_MULTIPROCESSOR                = 1024
MAX_SHARED_MEMORY_PER_BLOCK                   = 49152
MAX_SHARED_MEMORY_PER_MULTIPROCESSOR          = 65536
MAX_REGISTERS_PER_BLOCK                       = 65536
MAX_REGISTERS_PER_MULTIPROCESSOR              = 65536
CLOCK_RATE                                    = 1590000
MEMORY_CLOCK_RATE                             = 5001000
GLOBAL_MEMORY_

In [4]:
from numba.core.utils import threadsafe_cached_property
@cuda.jit
def un_kernel(my_array):
  pass

my_array = np.ones(320)
threads_per_block = 32
my_grid = 1000

un_kernel[my_grid, threads_per_block](my_array)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [6]:
import math
@cuda.jit
def kernel_doblador(my_array):
  position = cuda.threadIdx.x + cuda.blockIdx.x*cuda.blockDim.x
  #position = cuda.grid(1)
  if position < my_array.size:
    my_array[position] *= 2


my_array = np.ones(321)
# copiando el arreglo al dispositivo/device (GPU)
device_array = cuda.to_device(my_array)
threads_per_block = 32
grid =  math.ceil(my_array.size / threads_per_block)

kernel_doblador[grid, threads_per_block](device_array)
host_array = device_array.copy_to_host()
print(host_array)

[2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2.]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 11 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [8]:
@cuda.jit
def sum_kernel(a, b, c):
  position = cuda.grid(1)
  if position < a.size:
    c[position] = a[position] + b[position]

VECTOR_SIZE = 321
a = np.ones(VECTOR_SIZE)
b = np.ones(VECTOR_SIZE)
c = np.zeros(VECTOR_SIZE)

# Copiando al device
device_a = cuda.to_device(a)
device_b = cuda.to_device(b)
#device_c = cuda.device_array_like(c)
device_c = cuda.device_array(VECTOR_SIZE)


threads_per_block = 32
my_grid = math.ceil(a.size / threads_per_block)
sum_kernel[my_grid, threads_per_block](device_a, device_b, device_c)

print(device_c.copy_to_host())

[2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.
 2. 2. 2. 2. 2. 2. 2. 2. 2.]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 11 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


# Rejilla asociada a Renglones

In [9]:
np.set_printoptions(suppress=True)

@cuda.jit
def addition_matrix_kernel(a, b, c):
  i = cuda.grid(1)
  if i < a.shape[0]:
    for j in range(a.shape[1]):
      c[i][j] = a[i][j] + b[i][j]

ROWS = 64
COLUMNS = 128
host_a = np.arange(ROWS*COLUMNS).reshape((ROWS, COLUMNS))
host_b = np.arange(ROWS*COLUMNS).reshape((ROWS, COLUMNS))


dev_a = cuda.to_device(host_a)
dev_b = cuda.to_device(host_b)
dev_c = cuda.device_array((ROWS, COLUMNS))

threads_per_block = 32
my_grid = math.ceil(ROWS / threads_per_block)
print(my_grid, threads_per_block)
addition_matrix_kernel[my_grid, threads_per_block](dev_a, dev_b, dev_c)
print(dev_c.copy_to_host())

2 32
[[    0.     2.     4. ...   250.   252.   254.]
 [  256.   258.   260. ...   506.   508.   510.]
 [  512.   514.   516. ...   762.   764.   766.]
 ...
 [15616. 15618. 15620. ... 15866. 15868. 15870.]
 [15872. 15874. 15876. ... 16122. 16124. 16126.]
 [16128. 16130. 16132. ... 16378. 16380. 16382.]]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


# Grid asociado a columnas

In [10]:
@cuda.jit
def addition_matrix_kernel2(a, b, c):
  j = cuda.grid(1)
  if j < a.shape[1]:
    for i in range(a.shape[0]):
      c[i][j] = a[i][j] + b[i][j]

ROWS = 64
COLUMNS = 128
host_a = np.arange(ROWS*COLUMNS).reshape((ROWS, COLUMNS))
host_b = np.arange(ROWS*COLUMNS).reshape((ROWS, COLUMNS))


dev_a = cuda.to_device(host_a)
dev_b = cuda.to_device(host_b)
dev_c = cuda.device_array((ROWS, COLUMNS))

threads_per_block = 32
my_grid = math.ceil(COLUMNS / threads_per_block)
print(my_grid, threads_per_block)
addition_matrix_kernel2[my_grid, threads_per_block](dev_a, dev_b, dev_c)
print(dev_c.copy_to_host())

4 32


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


[[    0.     2.     4. ...   250.   252.   254.]
 [  256.   258.   260. ...   506.   508.   510.]
 [  512.   514.   516. ...   762.   764.   766.]
 ...
 [15616. 15618. 15620. ... 15866. 15868. 15870.]
 [15872. 15874. 15876. ... 16122. 16124. 16126.]
 [16128. 16130. 16132. ... 16378. 16380. 16382.]]


In [16]:
@cuda.jit
def suma_matrices(a, b, c):
  i = cuda.threadIdx.x + cuda.blockIdx.x*cuda.blockDim.x
  j = cuda.threadIdx.y + cuda.blockIdx.y*cuda.blockDim.y
  #i, j = cuda.grid(2)
  if i < a.shape[0] and j < a.shape[1]:
    c[i][j] = a[i][j] + b[i][j]

ROWS = 320
COLUMNS = 1280
host_a = np.arange(ROWS*COLUMNS).reshape((ROWS, COLUMNS))
host_b = np.arange(ROWS*COLUMNS).reshape((ROWS, COLUMNS))

dev_a = cuda.to_device(host_a)
dev_b = cuda.to_device(host_b)
dev_c = cuda.device_array((ROWS, COLUMNS))

block = (16, 32)

num_blocks_x = math.ceil( host_a.shape[0] / block[0])#  1280 / 32
num_blocks_y = math.ceil( host_a.shape[1] / block[1])#  320 / 16
grid = (num_blocks_x, num_blocks_y)
suma_matrices[grid, block](dev_a, dev_b, dev_c)
resultado = dev_c.copy_to_host()
print(resultado)

[[     0.      2.      4. ...   2554.   2556.   2558.]
 [  2560.   2562.   2564. ...   5114.   5116.   5118.]
 [  5120.   5122.   5124. ...   7674.   7676.   7678.]
 ...
 [811520. 811522. 811524. ... 814074. 814076. 814078.]
 [814080. 814082. 814084. ... 816634. 816636. 816638.]
 [816640. 816642. 816644. ... 819194. 819196. 819198.]]


In [12]:
320 / 16

20.0